In [6]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [7]:
df = pd.read_excel("DF_FINAL.xlsx")

df.head()

calles_base = [
        "Avenida Rafaela Ybarra",
        "Avenida de Oporto",
        "Calle Alcocer",
        "Calle Embajadores",
        "Calle Méndez Álvaro",
        "Calle Ronda de Valencia",
        "Calle Silvano",
        "Calle Toledo",
        "Calle de Gran Vía de San Francisco",
        "Calle de Marcelo Usera",
        "Paseo de Extremadura",
        "Paseo de Santa María de la Cabeza"
]

calles_sur = []

for base in calles_base:
    for col in df.columns:
        if base in col:
            calles_sur.append(col)

L = 3  # número de lags

X = pd.DataFrame()

for street in calles_sur:
    for lag in range(1, L + 1):
        X[f"{street}_t-{lag}"] = df[street].shift(lag)

# Variables externas reales
X["AEMET_tmed"] = df["AEMET_tmed"]
X["AEMET_prec"] = df["AEMET_prec"]
X["dia_semana"] = df["dia_semana"]
X["festivo"] = df["festivo"]

# Convertir variables categóricas a numéricas
X["dia_semana"] = X["dia_semana"].map({
    "lunes": 0,
    "martes": 1,
    "miércoles": 2,
    "jueves": 3,
    "viernes": 4,
    "sábado": 5,
    "domingo": 6
})

X["festivo"] = X["festivo"].map({
    "no": 0,
    "sí": 1
})


y_col = [col for col in df.columns if "Calle Embajadores" in col][0]
y = df[y_col]

# Limpiar NaNs
X = X.dropna()
y = y.loc[X.index]

In [8]:
# División temporal (80% train, 20% test)
split = int(len(X) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

print(X_train.shape, X_test.shape)

(15376, 73) (3845, 73)


In [9]:
with pd.ExcelWriter("dataset_XGBoost_Embajadores.xlsx") as writer:
    X_train.to_excel(writer, sheet_name="X_train")
    X_test.to_excel(writer, sheet_name="X_test")
    y_train.to_excel(writer, sheet_name="y_train")
    y_test.to_excel(writer, sheet_name="y_test")

In [10]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    gamma=0.1,
    random_state=42
)

model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=0.1, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=400,
             n_jobs=None, num_parallel_tree=None, ...)

In [17]:
y_pred = model.predict(X_test)
# Predicción próxima hora
X_next = X.iloc[[-1]]
y_next = model.predict(X_next)

print("Predicción tráfico en Embajadores (E-O) próxima hora:", y_next[0])

Predicción tráfico en Embajadores (E-O) próxima hora: 341.402


In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

mask = y_test != 0
mape = np.mean(np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])) * 100

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")

MAE: 50.31
RMSE: 67.99
MAPE: 10.29%


In [13]:
importances = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importances.head(10))

Calle Silvano_E-O_t-1                         0.509232
Calle Silvano_O-E_t-1                         0.144498
Calle Méndez Álvaro_N-S_t-1                   0.057934
Paseo de Santa María de la Cabeza_S-N_t-1     0.052711
Calle Méndez Álvaro_S-N_t-1                   0.045460
Calle Méndez Álvaro_S-N_t-2                   0.044335
Calle Embajadores_E-O_t-1                     0.017910
Calle Alcocer_E-O_t-3                         0.016539
Calle de Gran Vía de San Francisco_S-N_t-1    0.012844
Avenida Rafaela Ybarra_N-S_t-2                0.011510
dtype: float32


In [14]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=0,
    n_jobs=-1,
    scoring="neg_mean_absolute_error"
)

perm_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)

perm_df.head(10)


,feature,importance
33,Calle Silvano_E-O_t-1,85.519087
15,Calle Embajadores_E-O_t-1,65.882461
66,Paseo de Santa María de la Cabeza_S-N_t-1,13.707567
21,Calle Méndez Álvaro_N-S_t-1,11.304198
24,Calle Méndez Álvaro_S-N_t-1,9.188045
36,Calle Silvano_O-E_t-1,4.601869
34,Calle Silvano_E-O_t-2,4.529269
48,Calle de Gran Vía de San Francisco_S-N_t-1,2.589594
42,Calle Toledo_S-N_t-1,1.888541
71,dia_semana,1.047580


In [15]:
top_k = 8
top_features = perm_df.head(top_k)
top_features


,feature,importance
33,Calle Silvano_E-O_t-1,85.519087
15,Calle Embajadores_E-O_t-1,65.882461
66,Paseo de Santa María de la Cabeza_S-N_t-1,13.707567
21,Calle Méndez Álvaro_N-S_t-1,11.304198
24,Calle Méndez Álvaro_S-N_t-1,9.188045
36,Calle Silvano_O-E_t-1,4.601869
34,Calle Silvano_E-O_t-2,4.529269
48,Calle de Gran Vía de San Francisco_S-N_t-1,2.589594


In [16]:
top_streets = (
    top_features["feature"]
    .str.split("_")
    .str[0]
    .value_counts()
)
top_streets

feature
Calle Silvano                         3
Calle Méndez Álvaro                   2
Calle Embajadores                     1
Paseo de Santa María de la Cabeza     1
Calle de Gran Vía de San Francisco    1
Name: count, dtype: int64